# 2단계: drop vs median 전처리 비교

## 이 단계의 핵심 변화
결측/이상치 처리에서 drop보다 median 방향이 더 안정적인지 비교한 단계입니다.

## 왜 점수가 좋아졌나
특수일과 튀는 값이 섞인 데이터라 평균/삭제보다 중앙값이 더 안정적이었습니다.

## 안내
이 파일은 다운로드 오류를 피하기 위해 새 이름으로 다시 만든 버전입니다.
코드 셀 안에는 기존 주석이 남아 있거나, 원본 설명 흐름을 유지합니다.


# 두 데이터셋 최종 비교 노트북
이 노트북은 다음 두 데이터셋을 **같은 피처 엔지니어링**, **같은 시간순 검증 방식**, **같은 모델(XGBoost / LightGBM / CatBoost)** 으로 평가한 뒤,  
마지막에 **한 표에서 바로 비교**할 수 있게 만든 최종 비교 노트북입니다.

비교 대상:
- `train_drop.csv`
- `train_median.csv`

평가 타깃:
- `중식계`
- `석식계`

평가 지표:
- `MAE`
- `RMSE`
- `NMAE(%)`

In [ ]:
# 필요한 패키지 설치가 안 되어 있으면 먼저 실행하세요.
# !pip install xgboost lightgbm catboost

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.unicode_minus"] = False

# 한글 폰트 설정 (WSL/Linux/Windows 대응)
import matplotlib as mpl
import matplotlib.font_manager as fm
mpl.rcParams.update(mpl.rcParamsDefault)
sns.reset_defaults()

available_fonts = sorted(set(f.name for f in fm.fontManager.ttflist))
font_candidates = [
    "NanumGothic",
    "NanumBarunGothic",
    "NanumSquare",
    "NanumSquareRound",
    "Noto Sans CJK KR",
    "Noto Sans KR",
    "Malgun Gothic",
    "AppleGothic"
]

selected_font = None
for font in font_candidates:
    if font in available_fonts:
        selected_font = font
        break

if selected_font is not None:
    mpl.rcParams["font.family"] = selected_font
    mpl.rcParams["font.sans-serif"] = [selected_font]
    mpl.rcParams["axes.unicode_minus"] = False
    sns.set_theme(
        style="whitegrid",
        rc={
            "font.family": selected_font,
            "font.sans-serif": [selected_font],
            "axes.unicode_minus": False,
        }
    )
    print("적용 폰트:", selected_font)
else:
    print("한글 폰트를 찾지 못했습니다. 영문 폰트로 진행합니다.")

from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
# ---------------------------
# 1. 파일 경로 자동 탐색
# ---------------------------
drop_candidates = [
    r"C:\Users\yuzhd\github\team\data\made\train_drop.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/train_drop.csv",
    r"/mnt/data/train_drop.csv",
    r"./train_drop.csv",
]

median_candidates = [
    r"C:\Users\yuzhd\github\team\data\made\train_median.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/train_median.csv",
    r"/mnt/data/train_median.csv",
    r"./train_median.csv",
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

drop_path = find_existing_path(drop_candidates)
median_path = find_existing_path(median_candidates)

print("train_drop 경로:", drop_path)
print("train_median 경로:", median_path)

if drop_path is None:
    raise FileNotFoundError("train_drop.csv 파일을 찾지 못했습니다.")
if median_path is None:
    raise FileNotFoundError("train_median.csv 파일을 찾지 못했습니다.")

In [ ]:
# ---------------------------
# 2. 데이터 로드
# ---------------------------
drop_df = pd.read_csv(drop_path, encoding="utf-8-sig")
median_df = pd.read_csv(median_path, encoding="utf-8-sig")

drop_df["일자"] = pd.to_datetime(drop_df["일자"])
median_df["일자"] = pd.to_datetime(median_df["일자"])

print("drop_df shape:", drop_df.shape)
print("median_df shape:", median_df.shape)

display(drop_df.head())
display(median_df.head())

In [ ]:
# ---------------------------
# 3. 피처 엔지니어링 함수
# ---------------------------
def add_features(df):
    df = df.copy()
    df = df.sort_values("일자").reset_index(drop=True)

    # 날짜 파생
    df["year"] = df["일자"].dt.year
    df["month"] = df["일자"].dt.month
    df["day"] = df["일자"].dt.day
    df["weekday_num"] = df["일자"].dt.weekday
    df["weekofyear"] = df["일자"].dt.isocalendar().week.astype(int)

    month_end = df["일자"] + pd.offsets.MonthEnd(0)
    df["days_to_month_end"] = (month_end - df["일자"]).dt.days

    df["is_month_start_part"] = (df["day"] <= 5).astype(int)
    df["is_month_end_part"] = (df["days_to_month_end"] <= 5).astype(int)
    df["is_wed"] = (df["weekday_num"] == 2).astype(int)
    df["is_fri"] = (df["weekday_num"] == 4).astype(int)
    df["is_covid"] = (df["일자"] >= pd.Timestamp("2020-01-01")).astype(int)

    # 운영 변수 파생
    df["working_estimate"] = (
        df["본사정원수"]
        - df["본사휴가자수"]
        - df["본사출장자수"]
        - df["현본사소속재택근무자수"]
    )

    df["absence_ratio"] = (
        df["본사휴가자수"] + df["본사출장자수"] + df["현본사소속재택근무자수"]
    ) / (df["본사정원수"] + 1)

    df["휴가자비율"] = df["본사휴가자수"] / (df["본사정원수"] + 1)
    df["출장자비율"] = df["본사출장자수"] / (df["본사정원수"] + 1)
    df["재택자비율"] = df["현본사소속재택근무자수"] / (df["본사정원수"] + 1)

    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])
    df["overtime_per_working"] = df["본사시간외근무명령서승인건수"] / (df["working_estimate"] + 1)
    df["has_overtime"] = (df["본사시간외근무명령서승인건수"] > 0).astype(int)

    # 석식 특수일
    if "석식메뉴" in df.columns:
        df["is_special_dinner_day"] = (df["석식메뉴"].astype(str) == "자기계발의날").astype(int)
    else:
        df["is_special_dinner_day"] = 0

    return df

drop_feat = add_features(drop_df)
median_feat = add_features(median_df)

print(drop_feat.shape, median_feat.shape)

In [ ]:
# ---------------------------
# 4. 공통 피처 / 타깃 / 검증 구간
# ---------------------------
feature_cols = [
    "본사정원수",
    "본사휴가자수",
    "본사출장자수",
    "본사시간외근무명령서승인건수",
    "현본사소속재택근무자수",
    "year",
    "month",
    "day",
    "weekday_num",
    "weekofyear",
    "days_to_month_end",
    "is_month_start_part",
    "is_month_end_part",
    "is_wed",
    "is_fri",
    "is_covid",
    "working_estimate",
    "absence_ratio",
    "휴가자비율",
    "출장자비율",
    "재택자비율",
    "log_overtime",
    "overtime_per_working",
    "has_overtime",
    "is_special_dinner_day",
]

targets = ["중식계", "석식계"]

# 시간순 분할
split_date = pd.Timestamp("2020-11-01")

def split_train_valid(df):
    train_df = df[df["일자"] < split_date].copy()
    valid_df = df[df["일자"] >= split_date].copy()
    return train_df, valid_df

drop_train, drop_valid = split_train_valid(drop_feat)
median_train, median_valid = split_train_valid(median_feat)

print("drop train/valid:", drop_train.shape, drop_valid.shape)
print("median train/valid:", median_train.shape, median_valid.shape)

In [ ]:
# ---------------------------
# 5. 평가 함수
# ---------------------------
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def nmae(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

def evaluate_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "NMAE(%)": nmae(y_true, y_pred),
    }

In [ ]:
# ---------------------------
# 6. 모델 정의
# ---------------------------
def get_models():
    models = {
        "xgboost": XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="reg:squarederror",
            random_state=42
        ),
        "lightgbm": LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42
        ),
        "catboost": CatBoostRegressor(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            random_state=42,
            verbose=0
        )
    }
    return models

In [ ]:
# ---------------------------
# 7. 한 데이터셋 전체 실험 함수
# ---------------------------
def run_experiment(df_train, df_valid, dataset_name):
    results = []
    pred_store = {}

    X_train = df_train[feature_cols]
    X_valid = df_valid[feature_cols]

    models = get_models()

    for target in targets:
        y_train = df_train[target]
        y_valid = df_valid[target]

        for model_name, model in models.items():
            model.fit(X_train, y_train)
            pred = model.predict(X_valid)

            metrics = evaluate_regression(y_valid.values, pred)
            row = {
                "dataset": dataset_name,
                "target": target,
                "model": model_name,
                **metrics
            }
            results.append(row)

            pred_store[(dataset_name, target, model_name)] = {
                "y_true": y_valid.values,
                "y_pred": pred,
                "model_obj": model
            }

    result_df = pd.DataFrame(results).sort_values(["target", "NMAE(%)", "RMSE"])
    return result_df, pred_store

In [ ]:
# ---------------------------
# 8. 두 데이터셋 실험 실행
# ---------------------------
drop_result_df, drop_pred_store = run_experiment(drop_train, drop_valid, "train_drop")
median_result_df, median_pred_store = run_experiment(median_train, median_valid, "train_median")

display(drop_result_df)
display(median_result_df)

In [ ]:
# ---------------------------
# 9. 최종 비교 테이블 (한 표)
# ---------------------------
final_compare = pd.concat([drop_result_df, median_result_df], axis=0).reset_index(drop=True)
final_compare = final_compare.sort_values(["target", "NMAE(%)", "RMSE", "MAE"]).reset_index(drop=True)

print("[전체 비교 테이블]")
display(final_compare)

print("[타깃별 최고 결과]")
best_by_target = final_compare.groupby("target").first().reset_index()
display(best_by_target)

print("[데이터셋별 최고 결과]")
best_by_dataset_target = (
    final_compare.sort_values(["dataset", "target", "NMAE(%)"])
    .groupby(["dataset", "target"])
    .first()
    .reset_index()
)
display(best_by_dataset_target)

In [ ]:
# ---------------------------
# 10. Pivot 비교표
# ---------------------------
pivot_nmae = final_compare.pivot_table(
    index=["target", "model"],
    columns="dataset",
    values="NMAE(%)"
).sort_index()

pivot_rmse = final_compare.pivot_table(
    index=["target", "model"],
    columns="dataset",
    values="RMSE"
).sort_index()

pivot_mae = final_compare.pivot_table(
    index=["target", "model"],
    columns="dataset",
    values="MAE"
).sort_index()

print("[NMAE 비교표]")
display(pivot_nmae.style.format("{:.4f}"))

print("[RMSE 비교표]")
display(pivot_rmse.style.format("{:.4f}"))

print("[MAE 비교표]")
display(pivot_mae.style.format("{:.4f}"))

In [ ]:
# ---------------------------
# 11. 시각화: 데이터셋별/모델별 NMAE 비교
# ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, target in zip(axes, targets):
    plot_df = final_compare[final_compare["target"] == target].copy()
    sns.barplot(data=plot_df, x="model", y="NMAE(%)", hue="dataset", ax=ax)
    ax.set_title(f"{target} - 모델별 NMAE 비교")
    ax.set_xlabel("모델")
    ax.set_ylabel("NMAE(%)")

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------
# 12. 최고 모델의 실제값 vs 예측값
# ---------------------------
best_rows = (
    final_compare.sort_values(["target", "NMAE(%)"])
    .groupby("target")
    .first()
    .reset_index()
)

display(best_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (_, row) in zip(axes, best_rows.iterrows()):
    key = (row["dataset"], row["target"], row["model"])
    y_true = final_compare  # placeholder to keep style consistent
    store = drop_pred_store.get(key, None)
    if store is None:
        store = median_pred_store.get(key)

    ax.plot(store["y_true"], label="실제값")
    ax.plot(store["y_pred"], label="예측값")
    ax.set_title(f"{row['target']} 최고모델: {row['dataset']} / {row['model']}")
    ax.set_xlabel("검증셋 순서")
    ax.set_ylabel(row["target"])
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------
# 13. 피처 중요도 확인 함수
# ---------------------------
def get_feature_importance(model, feature_names):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        return pd.DataFrame({
            "feature": feature_names,
            "importance": importances
        }).sort_values("importance", ascending=False)
    return None

In [ ]:
# ---------------------------
# 14. 최고 모델별 피처 중요도 출력
# ---------------------------
for _, row in best_rows.iterrows():
    key = (row["dataset"], row["target"], row["model"])
    store = drop_pred_store.get(key, None)
    if store is None:
        store = median_pred_store.get(key)

    model_obj = store["model_obj"]
    imp_df = get_feature_importance(model_obj, feature_cols)

    print("=" * 80)
    print(f"[{row['target']}] 최고모델: {row['dataset']} / {row['model']}")
    if imp_df is not None:
        display(imp_df.head(15))

        plt.figure(figsize=(10, 6))
        sns.barplot(data=imp_df.head(15), x="importance", y="feature")
        plt.title(f"{row['target']} 최고모델 피처 중요도")
        plt.xlabel("importance")
        plt.ylabel("feature")
        plt.tight_layout()
        plt.show()
    else:
        print("이 모델은 feature_importances_를 직접 제공하지 않습니다.")

In [ ]:
# ---------------------------
# 15. 결과 저장
# ---------------------------
save_path = "final_dataset_model_comparison.csv"
final_compare.to_csv(save_path, index=False, encoding="utf-8-sig")
print("비교 결과 저장 완료:", save_path)

In [ ]:
# ---------------------------
# 16. 해석 가이드
# ---------------------------
print("""
해석 포인트:
1. target별로 NMAE(%)가 가장 낮은 모델이 우선 후보입니다.
2. 같은 모델이라도 train_drop / train_median 중 어느 데이터셋이 더 유리한지 비교합니다.
3. 중식계와 석식계의 최고 조합이 다를 수 있으므로, 타깃별로 따로 보는 것이 좋습니다.
4. 피처 중요도를 보고 working_estimate, overtime_per_working, is_special_dinner_day 등이 실제로 중요하게 작동하는지 확인합니다.
""")